Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large 
Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

How to create test datasets

How to run your RAG application on those datasets

How to measure your application's performance using different evaluation metrics
Overview

A typical RAG evaluation workflow consists of three main steps:

Creating a dataset with questions and their expected answers

Running your RAG application on those questions

Using evaluators to measure how well your application performed, looking at factors like:
Answer relevance

Answer accuracy

Retrieval quality

For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

Chatbot Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [7]:
LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"

In [9]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['b379b8e3-a719-4379-ba2c-cf414856145b',
  '9f7f573c-a5ec-4dac-bc60-6c15c9f28633',
  '30c24835-19e9-4246-8634-ec2671e365b6',
  '5cc1c9db-75bb-46b5-91d7-f827d56d0156',
  '24053246-4454-48b8-8e4d-9a8972f0d0f8'],
 'count': 5,
 'as_of': '2026-06-12T05:58:04.499132952Z'}

Define Metrics (LLM As A Judge)

In [ ]:
from langsmith import wrappers
 
openai_client=wrappers.wrap_openai(openai.OpenAI())

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      response=openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]
      ).choices[0].message.content

      return response == "CORRECT"